In [1]:
import json
import os

import pandas as pd
from openai import OpenAI

In [6]:
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = "gpt-5.4-mini"

print("OpenAI client ready.")

OpenAI client ready.


**Eval 4 — Temporal consistency**

In [7]:
with open("rag_summaries.json", "r") as f:
    rag_summaries = json.load(f)

print("Loaded summaries:", len(rag_summaries))
print(rag_summaries.keys())

Loaded summaries: 9
dict_keys(['Whole + Gemini', 'Whole + BGE', 'Whole + MedCPT', 'Fixed + Gemini', 'Fixed + BGE', 'Fixed + MedCPT', 'Section + Gemini', 'Section + BGE', 'Section + MedCPT'])


In [8]:
temporal_events_df = pd.read_csv(
    "temporal_candidate_events.csv"
)

In [9]:
TEMPORAL_CONSOLIDATION_PROMPT = """
You are consolidating clinical events for longitudinal temporal evaluation.

The source clinical notes have already been ordered by creation_timestamp.
The event's source note index therefore represents its authoritative
chronological position.

For each CURRENT candidate event, determine whether it represents:

KEEP:
- a genuinely new clinical event
- a new investigation, treatment, diagnosis, referral, or disposition
- a meaningful change in clinical state
- a new value/finding that represents clinical progression
- a treatment being started, stopped, changed, or meaningfully continued

REMOVE:
- the same clinical event already represented in EARLIER events
- historical information merely restated
- duplicated findings/results with no new clinical change
- administrative/non-clinically meaningful information

Important:
Do NOT remove genuine progression simply because it concerns the same
clinical concept.

Example:
"SpO2 89% on room air"
"SpO2 improved to 92% after oxygen"
"SpO2 later 94% on room air"
are separate meaningful states and should all be kept.

But repeated statements of the same NT-proBNP result of 4500 pg/mL,
without a new measurement or change, should not create multiple events.

Do not use dates written inside event text to establish chronology.
Use the supplied source note indices only.

Return valid JSON only:

{
  "decisions": [
    {
      "candidate_id": 0,
      "decision": "KEEP" or "REMOVE",
      "reason": "brief reason"
    }
  ]
}
"""

In [10]:
def consolidate_note_events(current_events, earlier_events):

    # Give each current candidate an ID so we can map
    # Nemotron's decision back to the original event.
    current_candidates = [
        {
            "candidate_id": i,
            "event": event
        }
        for i, event in enumerate(current_events)
    ]

    user_message = f"""
EARLIER CONSOLIDATED EVENTS:
{json.dumps(earlier_events, indent=2)}

CURRENT CANDIDATE EVENTS:
{json.dumps(current_candidates, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_CONSOLIDATION_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    # Get the model output safely.
    raw_output = response.choices[0].message.content

    if raw_output is None or not raw_output.strip():
        raise ValueError(
            "Nemotron returned an empty response."
        )

    # Remove Markdown fences if the model returns ```json ... ```
    raw_output = raw_output.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.replace("```json", "")
        raw_output = raw_output.replace("```", "")
        raw_output = raw_output.strip()

    result = json.loads(raw_output)

    return result["decisions"]

In [11]:
test_consolidated_events = []
test_decisions = []

for note_index in [0, 1, 2]:

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"] == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in test_consolidated_events
        ]
    )

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        record = {
            "source_note_index": note_index,
            "event": event_text,
            "decision": decision["decision"],
            "reason": decision["reason"]
        }

        test_decisions.append(record)

        if decision["decision"] == "KEEP":
            test_consolidated_events.append({
                "source_note_index": note_index,
                "event": event_text
            })


for item in test_decisions:
    print(
        f"NOTE {item['source_note_index']} | "
        f"{item['decision']} | "
        f"{item['event']}"
    )
    print("   Reason:", item["reason"])

NOTE 0 | KEEP | Patient presented with progressive breathlessness
   Reason: New presenting symptom indicating current clinical state
NOTE 0 | KEEP | Triage category 2 assigned for high-risk, potentially life-threatening condition
   Reason: New triage assessment reflecting acuity
NOTE 0 | KEEP | ED diagnosis of chronic thromboembolic pulmonary hypertension established
   Reason: New diagnosis established in ED
NOTE 0 | KEEP | Oxygen saturation recorded at 89% on room air
   Reason: New objective hypoxemia finding
NOTE 0 | KEEP | Low-flow oxygen therapy initiated
   Reason: New treatment started
NOTE 0 | KEEP | Admission to Respiratory High Dependency Unit planned
   Reason: New disposition/admission plan
NOTE 1 | REMOVE | Patient presented with progressive breathlessness
   Reason: Same presenting symptom already captured earlier.
NOTE 1 | REMOVE | Diagnosed with chronic thromboembolic pulmonary hypertension
   Reason: Duplicate of the established chronic thromboembolic pulmonary hype

In [12]:
consolidated_events = []
consolidation_decisions = []

for note_index in sorted(
    temporal_events_df["source_note_index"].unique()
):

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"] == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in consolidated_events
        ]
    )

    kept_count = 0

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        decision_record = {
            "source_note_index": note_index,
            "event": event_text,
            "decision": decision["decision"],
            "reason": decision["reason"]
        }

        consolidation_decisions.append(decision_record)

        if decision["decision"] == "KEEP":
            consolidated_events.append({
                "source_note_index": note_index,
                "event": event_text
            })
            kept_count += 1

    print(
        f"NOTE {note_index:02d} | "
        f"{len(current_events)} candidates | "
        f"{kept_count} kept"
    )

    # Save progress after every note.
    pd.DataFrame(consolidated_events).to_csv(
        "temporal_consolidated_events_gpt54mini.csv",
        index=False
    )

    pd.DataFrame(consolidation_decisions).to_csv(
        "temporal_consolidation_decisions_gpt54mini.csv",
        index=False
    )

print("\nDONE")
print("Final consolidated events:", len(consolidated_events))

NOTE 00 | 6 candidates | 6 kept
NOTE 01 | 6 candidates | 2 kept
NOTE 02 | 4 candidates | 1 kept
NOTE 03 | 7 candidates | 4 kept
NOTE 04 | 12 candidates | 8 kept
NOTE 05 | 12 candidates | 4 kept
NOTE 06 | 7 candidates | 2 kept
NOTE 07 | 4 candidates | 3 kept
NOTE 08 | 2 candidates | 1 kept
NOTE 09 | 8 candidates | 7 kept
NOTE 10 | 6 candidates | 0 kept
NOTE 11 | 3 candidates | 3 kept
NOTE 12 | 9 candidates | 6 kept
NOTE 13 | 4 candidates | 3 kept
NOTE 14 | 4 candidates | 4 kept
NOTE 15 | 5 candidates | 1 kept
NOTE 16 | 8 candidates | 2 kept
NOTE 17 | 6 candidates | 3 kept
NOTE 18 | 9 candidates | 3 kept
NOTE 19 | 6 candidates | 2 kept
NOTE 20 | 1 candidates | 1 kept
NOTE 21 | 7 candidates | 5 kept
NOTE 22 | 6 candidates | 0 kept
NOTE 23 | 4 candidates | 1 kept
NOTE 24 | 5 candidates | 4 kept
NOTE 25 | 7 candidates | 4 kept
NOTE 26 | 6 candidates | 1 kept
NOTE 27 | 8 candidates | 4 kept
NOTE 28 | 5 candidates | 1 kept
NOTE 29 | 2 candidates | 1 kept
NOTE 30 | 5 candidates | 2 kept
NOTE 3

In [13]:
SUMMARY_EVENT_MAPPING_PROMPT = """
You are aligning a generated longitudinal clinical summary with a
reference timeline of clinical events.

The reference events are already in authoritative chronological order,
based on the source clinical notes' creation_timestamp.

For each reference event, determine whether the underlying clinical event
is represented in the generated summary.

MATCH:
- The summary clearly expresses the same underlying clinical event.
- Paraphrasing is allowed.
- Exact wording is not required.
- A summary sentence may represent more than one reference event.

NO_MATCH:
- The event is absent from the summary.
- The summary only contains a vague related statement that does not
  establish the reference event.
- The clinical content is materially different.

Important:
- Do NOT judge whether the summary event occurs in the correct chronological
  position. Your task is only to identify correspondence and where the
  matched event appears in the summary.
- Do NOT penalize omissions.
- Do NOT judge hallucinations or unsupported claims.
- Do NOT use outside medical knowledge.
- Do NOT use dates inside the reference events to change their reference
  chronology.
- Match only on the clinical content of the event.

The generated summary is divided into numbered units in the order they
appear. For every matched reference event, return the number of the
summary unit that represents it.

Return valid JSON only:

{
  "matches": [
    {
      "reference_event_id": 0,
      "match": "MATCH",
      "summary_unit": 3
    },
    {
      "reference_event_id": 1,
      "match": "NO_MATCH",
      "summary_unit": null
    }
  ]
}
"""

In [14]:
import re


def split_summary_into_units(summary):
    """
    Split a generated summary into ordered text units.

    Each unit keeps its original position in the summary so that
    we can later measure the order of matched clinical events.
    """

    # Normalize whitespace while preserving the text itself.
    text = re.sub(r"\s+", " ", summary.strip())

    # Split primarily at sentence boundaries.
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # Remove empty pieces.
    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    # Give every unit an order number.
    units = [
        {
            "summary_unit": i,
            "text": sentence
        }
        for i, sentence in enumerate(sentences, start=1)
    ]

    return units

In [15]:
summary_units = {
    config: split_summary_into_units(summary)
    for config, summary in rag_summaries.items()
}

for config, units in summary_units.items():
    print(f"{config}: {len(units)} units")

Whole + Gemini: 43 units
Whole + BGE: 33 units
Whole + MedCPT: 28 units
Fixed + Gemini: 35 units
Fixed + BGE: 33 units
Fixed + MedCPT: 36 units
Section + Gemini: 35 units
Section + BGE: 25 units
Section + MedCPT: 52 units


In [16]:
reference_timeline = [
    {
        "reference_event_id": i,
        "source_note_index": event["source_note_index"],
        "event": event["event"]
    }
    for i, event in enumerate(consolidated_events)
]

print("Reference events:", len(reference_timeline))

# Quick look at the beginning
for event in reference_timeline[:5]:
    print(event)

Reference events: 116
{'reference_event_id': 0, 'source_note_index': np.int64(0), 'event': 'Patient presented with progressive breathlessness'}
{'reference_event_id': 1, 'source_note_index': np.int64(0), 'event': 'Triage category 2 assigned for high-risk, potentially life-threatening condition'}
{'reference_event_id': 2, 'source_note_index': np.int64(0), 'event': 'ED diagnosis of chronic thromboembolic pulmonary hypertension established'}
{'reference_event_id': 3, 'source_note_index': np.int64(0), 'event': 'Oxygen saturation recorded at 89% on room air'}
{'reference_event_id': 4, 'source_note_index': np.int64(0), 'event': 'Low-flow oxygen therapy initiated'}


In [18]:
def map_reference_events_to_summary(reference_timeline, units):

    # Send numbered summary units so the model only has to
    # identify correspondence and location.
    summary_for_model = [
        {
            "summary_unit": unit["summary_unit"],
            "text": unit["text"]
        }
        for unit in units
    ]

    user_message = f"""
REFERENCE TIMELINE:
{json.dumps(reference_timeline, indent=2)}

GENERATED SUMMARY UNITS:
{json.dumps(summary_for_model, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": SUMMARY_EVENT_MAPPING_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(
        response.choices[0].message.content
    )

    return result["matches"]

In [20]:
reference_timeline = [
    {
        "reference_event_id": int(i),
        "source_note_index": int(event["source_note_index"]),
        "event": event["event"]
    }
    for i, event in enumerate(consolidated_events)
]

print("Reference events:", len(reference_timeline))

Reference events: 116


In [21]:
test_config = "Whole + Gemini"

test_matches = map_reference_events_to_summary(
    reference_timeline,
    summary_units[test_config]
)

print("Returned decisions:", len(test_matches))

matched = [
    item for item in test_matches
    if item["match"] == "MATCH"
]

print("Matched reference events:", len(matched))

print("\nFirst 10 matches:")
for item in matched[:10]:
    print(item)

Returned decisions: 116
Matched reference events: 80

First 10 matches:
{'reference_event_id': 0, 'match': 'MATCH', 'summary_unit': 1}
{'reference_event_id': 2, 'match': 'MATCH', 'summary_unit': 6}
{'reference_event_id': 3, 'match': 'MATCH', 'summary_unit': 5}
{'reference_event_id': 4, 'match': 'MATCH', 'summary_unit': 7}
{'reference_event_id': 5, 'match': 'MATCH', 'summary_unit': 9}
{'reference_event_id': 6, 'match': 'MATCH', 'summary_unit': 7}
{'reference_event_id': 7, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 9, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 10, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 12, 'match': 'MATCH', 'summary_unit': 12}


In [22]:
temporal_mapping_results = {
    "Whole + Gemini": test_matches
}

for config, units in summary_units.items():

    # Already completed above.
    if config == "Whole + Gemini":
        continue

    print(f"Mapping: {config}")

    matches = map_reference_events_to_summary(
        reference_timeline,
        units
    )

    # Make sure GPT returned one decision for all 116 events.
    if len(matches) != len(reference_timeline):
        raise ValueError(
            f"{config}: expected {len(reference_timeline)} "
            f"decisions, got {len(matches)}"
        )

    temporal_mapping_results[config] = matches

    matched_count = sum(
        item["match"] == "MATCH"
        for item in matches
    )

    print(
        f"  Decisions: {len(matches)} | "
        f"Matched: {matched_count}"
    )

Mapping: Whole + BGE
  Decisions: 116 | Matched: 68
Mapping: Whole + MedCPT
  Decisions: 116 | Matched: 61
Mapping: Fixed + Gemini
  Decisions: 116 | Matched: 79
Mapping: Fixed + BGE
  Decisions: 116 | Matched: 61
Mapping: Fixed + MedCPT
  Decisions: 116 | Matched: 52
Mapping: Section + Gemini
  Decisions: 116 | Matched: 51
Mapping: Section + BGE
  Decisions: 116 | Matched: 67
Mapping: Section + MedCPT
  Decisions: 116 | Matched: 36


In [23]:
with open("temporal_summary_event_mappings.json", "w") as f:
    json.dump(
        temporal_mapping_results,
        f,
        indent=2
    )

print("Saved mappings:", len(temporal_mapping_results))

Saved mappings: 9


In [24]:
from itertools import combinations

import pandas as pd


def calculate_temporal_consistency(matches):
    """
    Calculate pairwise temporal-order accuracy.

    Only reference events that are actually represented in the summary
    are considered.

    Pairs mapped to the same summary unit are excluded because their
    relative order cannot be determined.
    """

    # Keep only matched reference events.
    matched_events = [
        {
            "reference_event_id": int(item["reference_event_id"]),
            "summary_unit": int(item["summary_unit"])
        }
        for item in matches
        if item["match"] == "MATCH"
        and item["summary_unit"] is not None
    ]

    # Reference event IDs already follow the authoritative
    # source chronology.
    matched_events = sorted(
        matched_events,
        key=lambda x: x["reference_event_id"]
    )

    correct_pairs = 0
    reversed_pairs = 0
    tied_pairs = 0

    for event_a, event_b in combinations(matched_events, 2):

        unit_a = event_a["summary_unit"]
        unit_b = event_b["summary_unit"]

        if unit_a < unit_b:
            correct_pairs += 1

        elif unit_a > unit_b:
            reversed_pairs += 1

        else:
            # Both events occur in the same summary sentence/unit,
            # so their internal ordering cannot be determined.
            tied_pairs += 1

    comparable_pairs = correct_pairs + reversed_pairs

    if comparable_pairs > 0:
        temporal_score = (
            correct_pairs / comparable_pairs
        ) * 100
    else:
        temporal_score = None

    return {
        "matched_events": len(matched_events),
        "correct_pairs": correct_pairs,
        "reversed_pairs": reversed_pairs,
        "tied_pairs_excluded": tied_pairs,
        "comparable_pairs": comparable_pairs,
        "temporal_consistency": temporal_score
    }

In [25]:
temporal_results = []

for config, matches in temporal_mapping_results.items():

    result = calculate_temporal_consistency(matches)

    temporal_results.append({
        "configuration": config,
        **result
    })

temporal_results_df = pd.DataFrame(temporal_results)

temporal_results_df = temporal_results_df.sort_values(
    "temporal_consistency",
    ascending=False
).reset_index(drop=True)

print(
    temporal_results_df[
        [
            "configuration",
            "matched_events",
            "correct_pairs",
            "reversed_pairs",
            "tied_pairs_excluded",
            "comparable_pairs",
            "temporal_consistency"
        ]
    ].to_string(index=False)
)

   configuration  matched_events  correct_pairs  reversed_pairs  tied_pairs_excluded  comparable_pairs  temporal_consistency
  Fixed + Gemini              79           2576             304                  201              2880             89.444444
     Whole + BGE              68           1895             309                   74              2204             85.980036
  Whole + MedCPT              61           1470             267                   93              1737             84.628670
   Section + BGE              67           1700             356                  155              2056             82.684825
     Fixed + BGE              61           1436             323                   71              1759             81.637294
Section + Gemini              51            944             257                   74              1201             78.601166
  Fixed + MedCPT              52           1002             281                   43              1283             78.098207


In [26]:
temporal_results_df.to_csv(
    "rag_temporal_consistency_results.csv",
    index=False
)

print("Saved temporal consistency results.")

Saved temporal consistency results.
